# Gradio & ChatInterface

**Week 2 Day 2 & Day 3 - Learning Lab**

Building UIs for LLM applications with Gradio and conversational AI with ChatInterface.

## Intent

Learn to:
- Create basic Gradio interfaces quickly
- Implement streaming responses with generators
- Build multi-model UIs with class-based design
- Use different component types (Textbox, Dropdown, Markdown)
- Build conversational AI with ChatInterface (Day 3)
- Manage conversation history and system messages (Day 3)
- Share and deploy Gradio apps

## Expected Insights

- Gradio is perfect for demos, prototypes, and MVPs
- Streaming requires generator pattern (`yield` not `return`)
- Class-based design with model registry simplifies multi-model UIs
- ChatInterface is purpose-built for conversations (Day 3)
- System messages control personality and behavior (Day 3)
- Pattern: Use Gradio for fast prototyping, custom web for production


In [ ]:
# Setup
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


## Experiment 1: Basic Gradio Interface

Create a simple text input → LLM → text output interface.

**Key Pattern:** `gr.Interface(fn=function, inputs=[...], outputs=[...])`


In [ ]:
# Basic Gradio interface
def simple_llm(prompt):
    """Simple LLM call - returns complete response."""
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    return response.choices[0].message.content

# Create interface
interface = gr.Interface(
    fn=simple_llm,
    inputs=gr.Textbox(label="Your message:", lines=5),
    outputs=gr.Textbox(label="Response:", lines=10),
    title="Simple LLM Chat",
    examples=["Hello!", "Explain transformers"],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# interface.launch()


## Experiment 2: Streaming with Generators

**Key Pattern:** Use `yield` keyword for streaming responses.

**Why:** Better UX - users see responses as they generate, not all at once.


In [ ]:
# Streaming LLM with generator pattern
def stream_llm(prompt):
    """Stream LLM response - yields incremental updates."""
    messages = [{"role": "user", "content": prompt}]
    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True  # Enable streaming
    )
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  # Yield accumulated result (not return!)

# Create streaming interface
streaming_interface = gr.Interface(
    fn=stream_llm,
    inputs=gr.Textbox(label="Your message:", lines=5),
    outputs=gr.Markdown(label="Response:"),  # Markdown for formatted output
    title="Streaming LLM Chat",
    examples=["Explain transformers", "Write a haiku"],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# streaming_interface.launch()


## Experiment 3: Multi-Model UI with Class-Based Design

**Key Pattern:** Model registry pattern - dictionary mapping model names to (client, model_name) tuples.

**Why:** Cleaner than separate functions per model, easy to extend.


In [ ]:
import requests

class MultiModelChat:
    """Unified chat interface supporting multiple LLM providers."""
    
    def __init__(self):
        """Initialize all model clients."""
        # OpenAI client
        self.openai_client = OpenAI()
        
        # Ollama client (check if available)
        self.ollama_available = False
        try:
            requests.get("http://localhost:11434/", timeout=2)
            self.ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
            self.ollama_available = True
        except:
            self.ollama_client = None
        
        # Model registry: maps display name to (client, model_name) tuple
        self.models = {
            "GPT": (self.openai_client, "gpt-4.1-mini"),
        }
        
        if self.ollama_available:
            self.models["Ollama"] = (self.ollama_client, "llama3.2")
    
    def chat(self, prompt, model_name):
        """
        Chat with selected model - streaming support.
        
        Args:
            prompt: User message
            model_name: Display name of model (e.g., "GPT", "Ollama")
        
        Yields:
            str: Incremental response chunks
        """
        if model_name not in self.models:
            yield f"Error: Model '{model_name}' not available."
            return
        
        client, model = self.models[model_name]
        
        messages = [{"role": "user", "content": prompt}]
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        result = ""
        for chunk in stream:
            result += chunk.choices[0].delta.content or ""
            yield result
    
    def get_available_models(self):
        """Return list of available model names."""
        return list(self.models.keys())


# Initialize chat
chat = MultiModelChat()

# Create multi-model interface
multi_model_interface = gr.Interface(
    fn=chat.chat,
    inputs=[
        gr.Textbox(label="Your message:", lines=5),
        gr.Dropdown(chat.get_available_models(), label="Select model", value="GPT")
    ],
    outputs=gr.Markdown(label="Response:"),
    title="Multi-Model Chat",
    examples=[
        ["Explain transformers", "GPT"],
        ["Write a haiku", "GPT"]
    ],
    flagging_mode="never"
)

# Launch (comment out to avoid auto-launching)
# multi_model_interface.launch()


## Experiment 4: Conversational AI with ChatInterface (Day 3)

**Key Pattern:** `gr.ChatInterface(fn=chat, type="messages")` - purpose-built for conversations.

**Why:** ChatInterface automatically manages conversation history, making multi-turn conversations much easier than using regular Interface.

**Key Differences:**
- **Interface:** General-purpose, you manage state yourself
- **ChatInterface:** Purpose-built for conversations, manages history automatically


In [ ]:
# ChatInterface for conversational AI
# System message defines personality and behavior
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off."

def chat(message, history):
    """
    Chat callback for ChatInterface.
    
    Args:
        message: Current user message
        history: List of previous messages in format [{"role": "...", "content": "..."}, ...]
    
    Returns:
        str: Assistant response (or yields for streaming)
    """
    # Convert Gradio history to API format
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # Build messages: system + history + new user message
    messages = [
        {"role": "system", "content": system_message},
    ] + history + [
        {"role": "user", "content": message}
    ]
    
    # Call API (non-streaming version)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    return response.choices[0].message.content

# Create ChatInterface
chat_interface = gr.ChatInterface(
    fn=chat,
    type="messages",  # Use messages format
    title="Clothes Store Assistant"
)

# Launch (comment out to avoid auto-launching)
# chat_interface.launch()


## Experiment 5: Streaming in ChatInterface (Day 3)

**Key Pattern:** Same generator pattern works in ChatInterface - use `yield` not `return`.

**Why:** Streaming feels more natural in conversations, users see responses as they generate.


In [ ]:
# Streaming ChatInterface
def chat_streaming(message, history):
    """
    Streaming chat callback for ChatInterface.
    
    Uses generator pattern - yields incremental updates.
    """
    # Convert Gradio history to API format
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # Build messages: system + history + new user message
    messages = [
        {"role": "system", "content": system_message},
    ] + history + [
        {"role": "user", "content": message}
    ]
    
    # Enable streaming
    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )
    
    # Accumulate and yield (same pattern as regular Interface)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response  # Yield accumulated result (not return!)

# Create streaming ChatInterface
streaming_chat_interface = gr.ChatInterface(
    fn=chat_streaming,
    type="messages",
    title="Streaming Clothes Store Assistant"
)

# Launch (comment out to avoid auto-launching)
# streaming_chat_interface.launch()


## Experiment 6: Dynamic System Message Modification (Day 3)

**Key Pattern:** Conditionally modify system message based on user input.

**Why:** Enables adaptive behavior - handle edge cases dynamically without cluttering base system message.


In [ ]:
# Dynamic system message modification
base_system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off."

def chat_dynamic(message, history):
    """
    Chat callback with dynamic system message modification.
    
    Modifies system message based on keywords in user message.
    """
    # Convert Gradio history to API format
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # Start with base system message
    relevant_system_message = base_system_message
    
    # Modify based on user input (keyword detection)
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    # Build messages with modified system message
    messages = [
        {"role": "system", "content": relevant_system_message},
    ] + history + [
        {"role": "user", "content": message}
    ]
    
    # Streaming response
    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )
    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

# Create ChatInterface with dynamic system message
dynamic_chat_interface = gr.ChatInterface(
    fn=chat_dynamic,
    type="messages",
    title="Adaptive Clothes Store Assistant"
)

# Launch (comment out to avoid auto-launching)
# dynamic_chat_interface.launch()


## Key Takeaways

### Gradio Basics
- **Simple interface:** `gr.Interface(fn=function, inputs=[...], outputs=[...])`
- **Component types:** Textbox (text), Dropdown (selection), Markdown (formatted output)
- **Launch options:** `share=True` (public link), `inbrowser=True` (auto-open), `auth=("user", "pass")` (password)
- **Examples:** Pre-populate UI with example inputs

### Streaming Pattern
- **Generator function:** Must use `yield`, not `return`
- **Gradio auto-detection:** Gradio automatically detects generator functions
- **Accumulate and yield:** Build result incrementally, yield after each chunk
- **Better UX:** Users see responses as they generate
- **Works in both:** Interface and ChatInterface support streaming

### Class-Based Multi-Model Design
- **Model registry:** Dictionary mapping names to (client, model_name) tuples
- **Unified method:** Single function works for all models in registry
- **Automatic detection:** Check availability, only include if ready
- **Easy extension:** Add new models by updating dictionary
- **Benefits:** No code duplication, cleaner architecture, self-documenting

### ChatInterface (Day 3)
- **Purpose-built:** `gr.ChatInterface(fn=chat, type="messages")` for conversations
- **Automatic history:** Gradio manages conversation history automatically
- **Callback signature:** `chat(message, history)` - simple and clean
- **History format:** Gradio provides history as list of `{"role": "...", "content": "..."}` dicts
- **Message conversion:** Must convert Gradio history to API format: `[system] + history + [user]`
- **System messages:** Perfect for personality, context, business rules, examples
- **Dynamic modification:** Conditionally modify system message based on user input
- **Pattern:** System message = personality + context, History = conversation

### Interface vs ChatInterface
- **Interface:** General-purpose, flexible, you manage state
- **ChatInterface:** Purpose-built for conversations, manages history automatically
- **When to use Interface:** Single-turn interactions, custom layouts, general I/O
- **When to use ChatInterface:** Multi-turn conversations, chat-like UIs
- **Pattern:** Use Interface for general I/O, ChatInterface for conversations

### When to Use Gradio
- **Perfect for:** Demos, prototypes, MVPs, internal tools
- **Not ideal for:** Production apps requiring full customization
- **Pattern:** Use Gradio for fast prototyping, custom web for production
